# 🦴 Batch FEM Graph Generator (Single Edge File)

This notebook processes **multiple FEM cases** where:
- Each case has its own **node features file** (different material properties per patient)
- All cases share the **same edge file** (same mesh topology)

## Features Included
- Centroid coordinates (x, y, z)
- Density, Stiffness
- **BC_Label** (Boundary Condition)
- **Loading_Label** (Loading Condition)
- Cort_Trab_Label (Cortical/Trabecular)
- Region_Label (Anatomical region)

## Output Naming
Output files are named exactly like input files:
- `Co_GNN_Inps_0010007803.csv` → `Co_GNN_Inps_0010007803_pyg.pkl`
- `Co_GNN_Inps_0010007803.csv` → `Co_GNN_Inps_0010007803_networkx.pkl`

---
## 1. Install Dependencies

In [ ]:
# Uncomment and run if needed
# !pip install pandas numpy networkx matplotlib tqdm
# !pip install torch torch-geometric

---
## 2. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import pickle
import os
import glob
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check for PyTorch Geometric
try:
    import torch
    from torch_geometric.data import Data
    PYTORCH_AVAILABLE = True
    print(f"✓ PyTorch version: {torch.__version__}")
    print(f"✓ PyTorch Geometric available")
except ImportError:
    PYTORCH_AVAILABLE = False
    print("⚠ PyTorch not available - will create NetworkX graphs only")
    print("  Install with: pip install torch torch-geometric")

print(f"\n✓ All imports successful!")

✓ PyTorch version: 2.0.1+cu118
✓ PyTorch Geometric available

✓ All imports successful!


---
## 3. Configuration

### UPDATE THESE PATHS TO MATCH YOUR SETUP!

In [6]:
# ============================================================
# CONFIGURATION - UPDATE THESE PATHS!
# ============================================================

# Directory containing your node CSV files
DATA_DIR = r"C:\Users\u232980\Bonestrength\Element_graph\1_16thJan\GNN-InpFol-AiO-260109"

# SINGLE edge file shared by all cases
EDGE_FILE = r"C:\Users\u232980\Bonestrength\Element_graph\1_16thJan\Bridges_filtered (375241 Common Faces).csv"

# Output directory for graphs
OUTPUT_DIR = "./graphs_v2"

# ============================================================
# FEATURE CONFIGURATION
# ============================================================

NUMERICAL_FEATURES = [
    'Centroid_x', 'Centroid_y', 'Centroid_z',  # Spatial coordinates
    'Density', 'Stiffness',                      # Material properties
    'BC_Label', 'Loading_Label'                  # Boundary & Loading conditions
]

CATEGORICAL_FEATURES = [
    'Cort_Trab_Label',  # Cortical/Trabecular
    'Region_Label'       # Anatomical region
]

# ============================================================
# CREATE OUTPUT DIRECTORY
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration:")
print("=" * 60)
print(f"Data directory:    {DATA_DIR}")
print(f"Edge file:         {EDGE_FILE}")
print(f"Output directory:  {OUTPUT_DIR}")
print(f"\nNumerical features: {NUMERICAL_FEATURES}")
print(f"Categorical features: {CATEGORICAL_FEATURES}")

Configuration:
Data directory:    C:\Users\u232980\Bonestrength\Element_graph\1_16thJan\GNN-InpFol-AiO-260109
Edge file:         C:\Users\u232980\Bonestrength\Element_graph\1_16thJan\Bridges_filtered (375241 Common Faces).csv
Output directory:  ./graphs_v2

Numerical features: ['Centroid_x', 'Centroid_y', 'Centroid_z', 'Density', 'Stiffness', 'BC_Label', 'Loading_Label']
Categorical features: ['Cort_Trab_Label', 'Region_Label']


---
## 4. Verify Paths

In [7]:
# Verify paths exist
print("Verifying paths...")
print("=" * 60)

# Check data directory
if os.path.exists(DATA_DIR):
    print(f"✓ Data directory exists")
    files_in_dir = os.listdir(DATA_DIR)
    csv_files = [f for f in files_in_dir if f.endswith('.csv')]
    print(f"  Found {len(csv_files)} CSV files")
else:
    print(f"✗ Data directory NOT FOUND!")
    print(f"  Please check: {DATA_DIR}")

# Check edge file
if os.path.exists(EDGE_FILE):
    print(f"✓ Edge file exists")
    edge_size = os.path.getsize(EDGE_FILE) / (1024 * 1024)
    print(f"  Size: {edge_size:.2f} MB")
else:
    print(f"✗ Edge file NOT FOUND!")
    print(f"  Please check: {EDGE_FILE}")

Verifying paths...
✓ Data directory exists
  Found 128 CSV files
✓ Edge file exists
  Size: 4.74 MB


---
## 5. Find All CSV Files and Create File Pairs

In [8]:
# ============================================================
# FIND ALL CSV FILES AND CREATE FILE PAIRS
# ============================================================

def get_base_name(filepath):
    """
    Get the base filename without extension.
    Example: 'Co_GNN_Inps_0010007803.csv' -> 'Co_GNN_Inps_0010007803'
    """
    basename = os.path.basename(filepath)
    name_without_ext = os.path.splitext(basename)[0]
    return name_without_ext


# Get ALL CSV files in the directory
all_csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))

print(f"Found {len(all_csv_files)} CSV files")

# Create pairs - using original filename as the output name
file_pairs = []
for csv_file in all_csv_files:
    base_name = get_base_name(csv_file)
    file_pairs.append({
        'base_name': base_name,           # For output filename
        'node_file': csv_file,
        'edge_file': EDGE_FILE
    })

print(f"Created {len(file_pairs)} file pairs")

print("\nFilename mapping (Input -> Output):")
print("-" * 70)
for pair in file_pairs[:5]:
    print(f"  {pair['base_name']}.csv")
    print(f"    -> {pair['base_name']}_networkx.pkl")
    print(f"    -> {pair['base_name']}_pyg.pkl")
    print()

if len(file_pairs) > 5:
    print(f"  ... and {len(file_pairs) - 5} more")

Found 128 CSV files
Created 128 file pairs

Filename mapping (Input -> Output):
----------------------------------------------------------------------
  Co_GNN_Inps_0010007803.csv
    -> Co_GNN_Inps_0010007803_networkx.pkl
    -> Co_GNN_Inps_0010007803_pyg.pkl

  Co_GNN_Inps_0012682800.csv
    -> Co_GNN_Inps_0012682800_networkx.pkl
    -> Co_GNN_Inps_0012682800_pyg.pkl

  Co_GNN_Inps_0013020209.csv
    -> Co_GNN_Inps_0013020209_networkx.pkl
    -> Co_GNN_Inps_0013020209_pyg.pkl

  Co_GNN_Inps_0013361705.csv
    -> Co_GNN_Inps_0013361705_networkx.pkl
    -> Co_GNN_Inps_0013361705_pyg.pkl

  Co_GNN_Inps_0013900409.csv
    -> Co_GNN_Inps_0013900409_networkx.pkl
    -> Co_GNN_Inps_0013900409_pyg.pkl

  ... and 123 more


---
## 6. Load Edge File (Once)

In [9]:
# Load edge file once (shared by all cases)
print("Loading edge file...")
print("=" * 60)

start_time = time.time()
edges_df = pd.read_csv(EDGE_FILE)
load_time = time.time() - start_time

print(f"✓ Loaded {len(edges_df):,} edges in {load_time:.2f} seconds")
print(f"\nEdge file columns: {list(edges_df.columns)}")
print(f"\nFirst 5 edges:")
print(edges_df.head())

Loading edge file...
✓ Loaded 375,240 edges in 0.05 seconds

Edge file columns: ['Element1', 'Element2']

First 5 edges:
   Element1  Element2
0         1         2
1         1         3
2         2         4
3         3         4
4         2         5


---
## 7. Load Sample Node File (Check Structure)

In [10]:
# Load first node file to check structure
sample_nodes_df = pd.read_csv(file_pairs[0]['node_file'])

print(f"Sample file: {os.path.basename(file_pairs[0]['node_file'])}")
print(f"Rows: {len(sample_nodes_df):,}")
print(f"Columns: {len(sample_nodes_df.columns)}")

print(f"\nColumn names:")
for col in sample_nodes_df.columns:
    print(f"  • {col}")

print(f"\nFirst 3 rows:")
sample_nodes_df.head(3)

Sample file: Co_GNN_Inps_0010007803.csv
Rows: 126,800
Columns: 14

Column names:
  • Element_ID
  • Centroid_x
  • Centroid_y
  • Centroid_z
  • Density
  • Stiffness
  • Cort_Trab_Label
  • Region_Label
  • BC_Label
  • Loading_Label
  • Height
  • Weight
  • Age
  • Side

First 3 rows:


,Element_ID,Centroid_x,Centroid_y,Centroid_z,Density,Stiffness,Cort_Trab_Label,Region_Label,BC_Label,Loading_Label,Height,Weight,Age,Side
0,1,124.016,67.596,808.306,638.976,3192.073,Trab,Head,0.0,0.0,157,71.5,71.43,1
1,2,123.956,67.473,806.989,648.814,3277.046,Trab,Head,0.0,0.0,157,71.5,71.43,1
2,3,124.615,66.614,808.358,641.061,3209.990,Trab,Head,0.0,0.0,157,71.5,71.43,1


In [11]:
# Check feature availability
print("Feature availability check:")
print("=" * 50)

print("\nNumerical features:")
for feat in NUMERICAL_FEATURES:
    exists = "✓" if feat in sample_nodes_df.columns else "✗ MISSING"
    print(f"  {exists} {feat}")

print("\nCategorical features:")
for feat in CATEGORICAL_FEATURES:
    exists = "✓" if feat in sample_nodes_df.columns else "✗ MISSING"
    print(f"  {exists} {feat}")

Feature availability check:

Numerical features:
  ✓ Centroid_x
  ✓ Centroid_y
  ✓ Centroid_z
  ✓ Density
  ✓ Stiffness
  ✓ BC_Label
  ✓ Loading_Label

Categorical features:
  ✓ Cort_Trab_Label
  ✓ Region_Label


In [12]:
# Check BC_Label and Loading_Label values
print("BC_Label distribution:")
print(sample_nodes_df['BC_Label'].value_counts())

print("\nLoading_Label distribution:")
print(sample_nodes_df['Loading_Label'].value_counts())

print("\nCort_Trab_Label distribution:")
print(sample_nodes_df['Cort_Trab_Label'].value_counts())

print("\nRegion_Label distribution:")
print(sample_nodes_df['Region_Label'].value_counts())

BC_Label distribution:
BC_Label
0.0    125416
0.6      1120
0.4       198
0.2        62
0.1         4
Name: count, dtype: int64

Loading_Label distribution:
Loading_Label
0.00    126452
0.95       276
0.65        42
0.35        30
Name: count, dtype: int64

Cort_Trab_Label distribution:
Cort_Trab_Label
Trab    99200
Cort    27600
Name: count, dtype: int64

Region_Label distribution:
Region_Label
Head    126800
Name: count, dtype: int64


In [13]:
print(sample_nodes_df['Cort_Trab_Label'].value_counts())
print(sample_nodes_df['Region_Label'].value_counts())

Cort_Trab_Label
Trab    99200
Cort    27600
Name: count, dtype: int64
Region_Label
Head    126800
Name: count, dtype: int64


---
## 8. Precompute Edge Index (Once)

In [14]:
# Precompute edge index using first node file as reference
print("Precomputing edge index...")
print("=" * 60)

# Get node IDs from first file
node_ids = sample_nodes_df['Element_ID'].values
id_to_idx = {node_id: idx for idx, node_id in enumerate(node_ids)}

print(f"Number of nodes: {len(node_ids):,}")

# Map edge endpoints to indices
print("Mapping edges to node indices...")
src_indices = edges_df['Element1'].map(id_to_idx).values
dst_indices = edges_df['Element2'].map(id_to_idx).values

# Check for any unmapped edges
valid_mask = ~(pd.isna(src_indices) | pd.isna(dst_indices))
num_valid = valid_mask.sum()
num_invalid = len(valid_mask) - num_valid

print(f"Valid edges: {num_valid:,}")
if num_invalid > 0:
    print(f"⚠ Invalid edges (missing nodes): {num_invalid:,}")

# Filter to valid edges
src_indices = src_indices[valid_mask].astype(int)
dst_indices = dst_indices[valid_mask].astype(int)

# Create undirected edge index (both directions)
edge_index_np = np.vstack([
    np.concatenate([src_indices, dst_indices]),
    np.concatenate([dst_indices, src_indices])
])

print(f"\nEdge index shape: {edge_index_np.shape}")
print(f"Total directed edges: {edge_index_np.shape[1]:,}")

# Convert to PyTorch tensor if available
if PYTORCH_AVAILABLE:
    precomputed_edge_index = torch.tensor(edge_index_np, dtype=torch.long)
    print(f"\n✓ PyTorch edge index created: {precomputed_edge_index.shape}")
else:
    precomputed_edge_index = None
    print("\n⚠ PyTorch not available - edge index stored as numpy array")

Precomputing edge index...
Number of nodes: 126,800
Mapping edges to node indices...
Valid edges: 375,240

Edge index shape: (2, 750480)
Total directed edges: 750,480

✓ PyTorch edge index created: torch.Size([2, 750480])


---
## 9. Define Processing Functions

In [15]:
def preprocess_features(nodes_df, numerical_cols, categorical_cols):
    """
    Preprocess node features:
    - Normalize numerical features (z-score)
    - One-hot encode categorical features
    
    Returns:
        features_df: DataFrame with processed features
        feature_names: List of feature column names
    """
    # Filter to columns that exist
    num_cols = [c for c in numerical_cols if c in nodes_df.columns]
    cat_cols = [c for c in categorical_cols if c in nodes_df.columns]
    
    # Normalize numerical features
    if num_cols:
        num_features = nodes_df[num_cols].copy()
        num_normalized = (num_features - num_features.mean()) / (num_features.std() + 1e-8)
    else:
        num_normalized = pd.DataFrame()
    
    # One-hot encode categorical features
    if cat_cols:
        cat_encoded = pd.get_dummies(nodes_df[cat_cols], prefix=cat_cols)
    else:
        cat_encoded = pd.DataFrame()
    
    # Combine all features
    features_df = pd.concat([num_normalized, cat_encoded], axis=1)
    features_df = features_df.astype(np.float64)
    
    return features_df, list(features_df.columns)


def build_networkx_graph(nodes_df, edges_df, features_df):
    """
    Build a NetworkX graph with node attributes.
    
    Returns:
        G: NetworkX graph
    """
    G = nx.Graph()
    
    # Add nodes with attributes
    for idx, row in nodes_df.iterrows():
        G.add_node(
            row['Element_ID'],
            x=row.get('Centroid_x', 0),
            y=row.get('Centroid_y', 0),
            z=row.get('Centroid_z', 0),
            density=row.get('Density', 0),
            stiffness=row.get('Stiffness', 0),
            bone_type=row.get('Cort_Trab_Label', 'Unknown'),
            region=row.get('Region_Label', 'Unknown'),
            bc_label=row.get('BC_Label', 0),
            loading_label=row.get('Loading_Label', 0),
            feature_vector=features_df.iloc[idx].values
        )
    
    # Add edges
    edges_list = list(zip(edges_df['Element1'], edges_df['Element2']))
    G.add_edges_from(edges_list)
    
    return G


def build_pyg_data(nodes_df, features_df, precomputed_edge_index):
    """
    Build PyTorch Geometric Data object using precomputed edge index.
    
    Returns:
        data: PyTorch Geometric Data object
    """
    if not PYTORCH_AVAILABLE:
        return None
    
    # Node features tensor
    x = torch.tensor(features_df.values, dtype=torch.float32)
    
    # Use precomputed edge index (clone to avoid modifying original)
    edge_index = precomputed_edge_index.clone()
    
    # Create Data object
    data = Data(x=x, edge_index=edge_index)
    data.num_nodes = len(nodes_df)
    data.num_edges = edge_index.shape[1]
    
    return data


def process_single_case(base_name, node_file, edges_df, precomputed_edge_index,
                        output_dir, numerical_cols, categorical_cols,
                        save_networkx=True, save_pyg=True):
    """
    Process a single FEM case and save graphs with original filename.
    
    Output files:
        - {base_name}_networkx.pkl
        - {base_name}_pyg.pkl
    """
    result = {
        'base_name': base_name,
        'status': 'success',
        'error': None,
        'num_nodes': 0,
        'num_edges': 0,
        'num_features': 0,
        'avg_degree': 0,
        'processing_time': 0,
        'networkx_file': None,
        'pyg_file': None
    }
    
    start_time = time.time()
    
    try:
        # Load node data
        nodes_df = pd.read_csv(node_file)
        
        # Preprocess features
        features_df, feature_names = preprocess_features(
            nodes_df, numerical_cols, categorical_cols
        )
        
        # Build NetworkX graph
        G = build_networkx_graph(nodes_df, edges_df, features_df)
        
        # Build PyG data
        pyg_data = build_pyg_data(nodes_df, features_df, precomputed_edge_index)
        
        # Calculate statistics
        result['num_nodes'] = G.number_of_nodes()
        result['num_edges'] = G.number_of_edges()
        result['num_features'] = len(feature_names)
        degrees = [d for n, d in G.degree()]
        result['avg_degree'] = np.mean(degrees) if degrees else 0
        result['feature_names'] = feature_names
        
        # Save NetworkX graph (using base_name)
        if save_networkx:
            nx_path = os.path.join(output_dir, f"{base_name}_networkx.pkl")
            with open(nx_path, 'wb') as f:
                pickle.dump(G, f)
            result['networkx_file'] = nx_path
        
        # Save PyG data (using base_name)
        if save_pyg and pyg_data is not None:
            pyg_path = os.path.join(output_dir, f"{base_name}_pyg.pkl")
            with open(pyg_path, 'wb') as f:
                pickle.dump(pyg_data, f)
            result['pyg_file'] = pyg_path
        
    except Exception as e:
        result['status'] = 'failed'
        result['error'] = str(e)
    
    result['processing_time'] = time.time() - start_time
    
    return result


print("✓ Processing functions defined")

✓ Processing functions defined


---
## 10. Test on Single Case

In [ ]:
# Test processing on first case
print("Testing on first case...")
print("=" * 60)

test_pair = file_pairs[0]
print(f"File: {test_pair['base_name']}.csv")

# Load node data
test_nodes_df = pd.read_csv(test_pair['node_file'])
print(f"Loaded {len(test_nodes_df):,} nodes")

# Preprocess features
test_features_df, feature_names = preprocess_features(
    test_nodes_df, NUMERICAL_FEATURES, CATEGORICAL_FEATURES
)
print(f"\nFeatures ({len(feature_names)} total):")
for i, name in enumerate(feature_names):
    print(f"  [{i:2d}] {name}")

# Build NetworkX graph
test_G = build_networkx_graph(test_nodes_df, edges_df, test_features_df)
print(f"\nNetworkX graph:")
print(f"  Nodes: {test_G.number_of_nodes():,}")
print(f"  Edges: {test_G.number_of_edges():,}")

# Build PyG data
if PYTORCH_AVAILABLE:
    test_pyg = build_pyg_data(test_nodes_df, test_features_df, precomputed_edge_index)
    print(f"\nPyTorch Geometric data:")
    print(f"  x shape: {test_pyg.x.shape}")
    print(f"  edge_index shape: {test_pyg.edge_index.shape}")

print("\n✓ Test successful!")

---
## 11. Process All Cases

In [16]:
# ============================================================
# PROCESS ALL CASES
# ============================================================

print("=" * 70)
print(f"  PROCESSING {len(file_pairs)} CASES")
print("=" * 70)
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"\nFilename format:")
print(f"  Input:  <filename>.csv")
print(f"  Output: <filename>_networkx.pkl")
print(f"          <filename>_pyg.pkl")
print("\n" + "-" * 70)

results = []
successful = 0
failed = 0

total_start = time.time()

# Process each case with progress bar
for pair in tqdm(file_pairs, desc="Processing cases"):
    result = process_single_case(
        base_name=pair['base_name'],          # Use original filename
        node_file=pair['node_file'],
        edges_df=edges_df,
        precomputed_edge_index=precomputed_edge_index,
        output_dir=OUTPUT_DIR,
        numerical_cols=NUMERICAL_FEATURES,
        categorical_cols=CATEGORICAL_FEATURES,
        save_networkx=True,
        save_pyg=True
    )
    
    results.append(result)
    
    if result['status'] == 'success':
        successful += 1
    else:
        failed += 1
        print(f"\n  ⚠ {pair['base_name']} failed: {result['error']}")

total_time = time.time() - total_start

# Summary
print("\n" + "=" * 70)
print("  PROCESSING COMPLETE!")
print("=" * 70)
print(f"\n  ✓ Successful: {successful}")
print(f"  ✗ Failed:     {failed}")
print(f"  Total time:   {total_time:.1f} seconds ({total_time/60:.1f} minutes)")
print(f"  Avg per case: {total_time/len(file_pairs):.2f} seconds")

  PROCESSING 128 CASES

Output directory: ./graphs_v2

Filename format:
  Input:  <filename>.csv
  Output: <filename>_networkx.pkl
          <filename>_pyg.pkl

----------------------------------------------------------------------


Processing cases: 100%|██████████| 128/128 [17:14<00:00,  8.08s/it]


  PROCESSING COMPLETE!

  ✓ Successful: 128
  ✗ Failed:     0
  Total time:   1034.2 seconds (17.2 minutes)
  Avg per case: 8.08 seconds


## Check whether the generated graphs are consistent: whether they have same number of features per node

In [18]:
"""
Script to verify all generated graphs have consistent number of node features
"""

import os
import pickle
import glob
from collections import Counter
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

GRAPHS_DIR = r"C:\Users\u232980\Bonestrength\Element_graph\1_16thJan\graphs_v2"  # Directory where your graphs are saved

# ============================================================
# CHECK FUNCTIONS
# ============================================================

def check_networkx_features(graph_files):
    """Check node feature consistency in NetworkX graphs"""
    print("=" * 70)
    print("  CHECKING NETWORKX GRAPHS")
    print("=" * 70)
    
    feature_counts = []
    feature_length_counts = []
    issues = []
    
    for i, graph_file in enumerate(graph_files):
        try:
            with open(graph_file, 'rb') as f:
                G = pickle.load(f)
            
            base_name = os.path.basename(graph_file)
            num_nodes = G.number_of_nodes()
            
            # Get feature vector from first node
            first_node = list(G.nodes())[0]
            feature_vector = G.nodes[first_node].get('feature_vector', None)
            
            if feature_vector is None:
                issues.append(f"{base_name}: No feature_vector attribute")
                continue
            
            num_features = len(feature_vector)
            feature_counts.append(num_features)
            
            # Check if all nodes have same feature length
            node_feature_lengths = set()
            for node in G.nodes():
                fv = G.nodes[node].get('feature_vector', None)
                if fv is not None:
                    node_feature_lengths.add(len(fv))
            
            if len(node_feature_lengths) > 1:
                issues.append(f"{base_name}: Inconsistent feature lengths within graph: {node_feature_lengths}")
            
            # Progress indicator
            if (i + 1) % 20 == 0:
                print(f"  Checked {i + 1}/{len(graph_files)} graphs...")
                
        except Exception as e:
            issues.append(f"{base_name}: Error loading - {str(e)}")
    
    return feature_counts, issues


def check_pyg_features(graph_files):
    """Check node feature consistency in PyTorch Geometric graphs"""
    print("\n" + "=" * 70)
    print("  CHECKING PYTORCH GEOMETRIC GRAPHS")
    print("=" * 70)
    
    feature_counts = []
    issues = []
    
    for i, graph_file in enumerate(graph_files):
        try:
            with open(graph_file, 'rb') as f:
                data = pickle.load(f)
            
            base_name = os.path.basename(graph_file)
            
            if not hasattr(data, 'x'):
                issues.append(f"{base_name}: No 'x' attribute (feature matrix)")
                continue
            
            # data.x shape is [num_nodes, num_features]
            num_nodes = data.x.shape[0]
            num_features = data.x.shape[1]
            
            feature_counts.append(num_features)
            
            # Progress indicator
            if (i + 1) % 20 == 0:
                print(f"  Checked {i + 1}/{len(graph_files)} graphs...")
                
        except Exception as e:
            issues.append(f"{base_name}: Error loading - {str(e)}")
    
    return feature_counts, issues


def print_summary(feature_counts, issues, graph_type):
    """Print summary statistics"""
    print("\n" + "-" * 70)
    print(f"  {graph_type.upper()} SUMMARY")
    print("-" * 70)
    
    if not feature_counts:
        print("  ⚠ No valid graphs found!")
        return
    
    # Count occurrences of each feature count
    count_distribution = Counter(feature_counts)
    
    print(f"\n  Total graphs checked: {len(feature_counts)}")
    print(f"\n  Feature count distribution:")
    for num_features, count in sorted(count_distribution.items()):
        percentage = (count / len(feature_counts)) * 100
        print(f"    {num_features} features: {count} graphs ({percentage:.1f}%)")
    
    # Check consistency
    unique_counts = len(count_distribution)
    
    if unique_counts == 1:
        print(f"\n  ✓ ALL GRAPHS CONSISTENT: {feature_counts[0]} features per node")
    else:
        print(f"\n  ✗ INCONSISTENT: {unique_counts} different feature counts found!")
        print(f"    Expected: {max(count_distribution, key=count_distribution.get)} features (most common)")
        print(f"    Range: {min(feature_counts)} - {max(feature_counts)} features")
    
    # Print issues
    if issues:
        print(f"\n  ⚠ Issues found ({len(issues)}):")
        for issue in issues[:10]:  # Show first 10
            print(f"    • {issue}")
        if len(issues) > 10:
            print(f"    ... and {len(issues) - 10} more issues")
    else:
        print(f"\n  ✓ No issues found")


def create_detailed_report(networkx_counts, pyg_counts, networkx_issues, pyg_issues, output_file):
    """Create a detailed CSV report"""
    
    # Get all graph files
    nx_files = sorted(glob.glob(os.path.join(GRAPHS_DIR, "*_networkx.pkl")))
    pyg_files = sorted(glob.glob(os.path.join(GRAPHS_DIR, "*_pyg.pkl")))
    
    report_data = []
    
    for nx_file, pyg_file in zip(nx_files, pyg_files):
        base_name = os.path.basename(nx_file).replace('_networkx.pkl', '')
        
        # Get NetworkX info
        try:
            with open(nx_file, 'rb') as f:
                G = pickle.load(f)
            nx_nodes = G.number_of_nodes()
            nx_edges = G.number_of_edges()
            first_node = list(G.nodes())[0]
            nx_features = len(G.nodes[first_node].get('feature_vector', []))
        except:
            nx_nodes = nx_edges = nx_features = None
        
        # Get PyG info
        try:
            with open(pyg_file, 'rb') as f:
                data = pickle.load(f)
            pyg_nodes = data.num_nodes
            pyg_edges = data.num_edges
            pyg_features = data.x.shape[1]
        except:
            pyg_nodes = pyg_edges = pyg_features = None
        
        report_data.append({
            'graph_name': base_name,
            'nx_nodes': nx_nodes,
            'nx_edges': nx_edges,
            'nx_features': nx_features,
            'pyg_nodes': pyg_nodes,
            'pyg_edges': pyg_edges,
            'pyg_features': pyg_features,
            'features_match': nx_features == pyg_features if (nx_features and pyg_features) else None
        })
    
    # Save to CSV
    df = pd.DataFrame(report_data)
    df.to_csv(output_file, index=False)
    
    return df


# ============================================================
# MAIN EXECUTION
# ============================================================

if __name__ == "__main__":
    
    print("\n" + "=" * 70)
    print("  GRAPH FEATURE CONSISTENCY CHECKER")
    print("=" * 70)
    print(f"\n  Graphs directory: {GRAPHS_DIR}\n")
    
    # Find all graph files
    networkx_files = sorted(glob.glob(os.path.join(GRAPHS_DIR, "*_networkx.pkl")))
    pyg_files = sorted(glob.glob(os.path.join(GRAPHS_DIR, "*_pyg.pkl")))
    
    print(f"  Found {len(networkx_files)} NetworkX graphs")
    print(f"  Found {len(pyg_files)} PyTorch Geometric graphs")
    
    if not networkx_files and not pyg_files:
        print("\n  ⚠ No graph files found! Check the GRAPHS_DIR path.")
        exit(1)
    
    # Check NetworkX graphs
    if networkx_files:
        nx_counts, nx_issues = check_networkx_features(networkx_files)
        print_summary(nx_counts, nx_issues, "NetworkX")
    
    # Check PyG graphs
    if pyg_files:
        pyg_counts, pyg_issues = check_pyg_features(pyg_files)
        print_summary(pyg_counts, pyg_issues, "PyTorch Geometric")
    
    # Create detailed report
    print("\n" + "=" * 70)
    print("  GENERATING DETAILED REPORT")
    print("=" * 70)
    
    report_file = os.path.join(GRAPHS_DIR, "feature_consistency_report.csv")
    df = create_detailed_report(
        nx_counts if networkx_files else [],
        pyg_counts if pyg_files else [],
        nx_issues if networkx_files else [],
        pyg_issues if pyg_files else [],
        report_file
    )
    
    print(f"\n  ✓ Detailed report saved: {report_file}")
    
    # Show sample of report
    print("\n  Sample from report (first 5 graphs):")
    print("-" * 70)
    print(df.head().to_string(index=False))
    
    # Final summary
    print("\n" + "=" * 70)
    print("  FINAL VERDICT")
    print("=" * 70)
    
    all_consistent = True
    
    if networkx_files and len(set(nx_counts)) > 1:
        all_consistent = False
        print("  ✗ NetworkX graphs have INCONSISTENT feature counts")
    elif networkx_files:
        print(f"  ✓ NetworkX graphs: {nx_counts[0]} features (consistent)")
    
    if pyg_files and len(set(pyg_counts)) > 1:
        all_consistent = False
        print("  ✗ PyTorch Geometric graphs have INCONSISTENT feature counts")
    elif pyg_files:
        print(f"  ✓ PyTorch Geometric graphs: {pyg_counts[0]} features (consistent)")
    
    if networkx_files and pyg_files:
        if nx_counts[0] != pyg_counts[0]:
            all_consistent = False
            print(f"  ✗ NetworkX and PyG feature counts DON'T MATCH!")
            print(f"    NetworkX: {nx_counts[0]} features")
            print(f"    PyG: {pyg_counts[0]} features")
        else:
            print(f"  ✓ NetworkX and PyG feature counts MATCH: {nx_counts[0]} features")
    
    print("\n" + "=" * 70)
    if all_consistent:
        print("  🎉 ALL GRAPHS HAVE CONSISTENT FEATURES!")
    else:
        print("  ⚠️  INCONSISTENCIES DETECTED - CHECK REPORT FOR DETAILS")
    print("=" * 70 + "\n")


  GRAPH FEATURE CONSISTENCY CHECKER

  Graphs directory: C:\Users\u232980\Bonestrength\Element_graph\1_16thJan\graphs_v2

  Found 128 NetworkX graphs
  Found 128 PyTorch Geometric graphs
  CHECKING NETWORKX GRAPHS
  Checked 20/128 graphs...
  Checked 40/128 graphs...
  Checked 60/128 graphs...
  Checked 80/128 graphs...
  Checked 100/128 graphs...
  Checked 120/128 graphs...

----------------------------------------------------------------------
  NETWORKX SUMMARY
----------------------------------------------------------------------

  Total graphs checked: 128

  Feature count distribution:
    10 features: 64 graphs (50.0%)
    12 features: 64 graphs (50.0%)

  ✗ INCONSISTENT: 2 different feature counts found!
    Expected: 10 features (most common)
    Range: 10 - 12 features

  ✓ No issues found

  CHECKING PYTORCH GEOMETRIC GRAPHS
  Checked 20/128 graphs...
  Checked 40/128 graphs...
  Checked 60/128 graphs...
  Checked 80/128 graphs...
  Checked 100/128 graphs...
  Checked 120/

In [21]:
# Compare a 10-feature vs 12-feature source file
df_10 = pd.read_csv(r'C:\Users\u232980\Bonestrength\Element_graph\1_16thJan\GNN-InpFol-AiO-260109\Co_GNN_Inps_0010007803.csv')
df_12 = pd.read_csv(r'C:\Users\u232980\Bonestrength\Element_graph\1_16thJan\GNN-InpFol-AiO-260109\Fx_GNN_Inps_45986.csv')

print(df_10['Region_Label'].unique())
print(df_10['Cort_Trab_Label'].unique())
print(df_12['Region_Label'].unique())
print(df_12['Cort_Trab_Label'].unique())

['Head']
['Trab' 'Cort']
['Troch' 'Head' 'Neck']
['Trab' 'Cort']


The problem is Fracture cases have 3 region labels- Troch,Head and Neck. So we should make the preprocessing consistent.

---
## 12. Save Summary Report

In [ ]:
# Create summary DataFrame
summary_data = []
for r in results:
    summary_data.append({
        'filename': r['base_name'],
        'status': r['status'],
        'num_nodes': r['num_nodes'],
        'num_edges': r['num_edges'],
        'num_features': r['num_features'],
        'avg_degree': round(r['avg_degree'], 2),
        'processing_time': round(r['processing_time'], 2),
        'networkx_file': os.path.basename(r['networkx_file']) if r['networkx_file'] else None,
        'pyg_file': os.path.basename(r['pyg_file']) if r['pyg_file'] else None,
        'error': r['error']
    })

summary_df = pd.DataFrame(summary_data)

# Save summary CSV
summary_path = os.path.join(OUTPUT_DIR, 'all_graphs_summary.csv')
summary_df.to_csv(summary_path, index=False)
print(f"✓ Saved summary to: {summary_path}")

# Display summary
print("\nSummary statistics:")
print(summary_df.describe())

In [ ]:
# View all cases
print(f"All {len(summary_df)} cases:")
summary_df

In [ ]:
# Check for failed cases
failed_cases = summary_df[summary_df['status'] == 'failed']

if len(failed_cases) > 0:
    print(f"⚠ {len(failed_cases)} cases failed:")
    print(failed_cases[['filename', 'error']])
else:
    print("✓ All cases processed successfully!")

---
## 13. Save Feature Names

In [ ]:
# Get feature names from first successful case
feature_names = None
for r in results:
    if r['status'] == 'success' and 'feature_names' in r:
        feature_names = r['feature_names']
        break

if feature_names:
    # Save feature names
    feature_path = os.path.join(OUTPUT_DIR, 'feature_names.pkl')
    with open(feature_path, 'wb') as f:
        pickle.dump(feature_names, f)
    print(f"✓ Saved feature names to: {feature_path}")
    
    print(f"\nFeatures ({len(feature_names)} total):")
    for i, name in enumerate(feature_names):
        print(f"  [{i:2d}] {name}")

---
## 14. Visualize Results

In [ ]:
# Filter successful cases
success_df = summary_df[summary_df['status'] == 'success']

if len(success_df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Nodes per case
    axes[0, 0].bar(range(len(success_df)), success_df['num_nodes'], color='steelblue', alpha=0.7)
    axes[0, 0].axhline(success_df['num_nodes'].mean(), color='red', linestyle='--', 
                       label=f"Mean: {success_df['num_nodes'].mean():,.0f}")
    axes[0, 0].set_xlabel('Case Index')
    axes[0, 0].set_ylabel('Number of Nodes')
    axes[0, 0].set_title('Nodes per Case')
    axes[0, 0].legend()
    
    # Plot 2: Edges per case
    axes[0, 1].bar(range(len(success_df)), success_df['num_edges'], color='coral', alpha=0.7)
    axes[0, 1].axhline(success_df['num_edges'].mean(), color='red', linestyle='--',
                       label=f"Mean: {success_df['num_edges'].mean():,.0f}")
    axes[0, 1].set_xlabel('Case Index')
    axes[0, 1].set_ylabel('Number of Edges')
    axes[0, 1].set_title('Edges per Case')
    axes[0, 1].legend()
    
    # Plot 3: Average degree
    axes[1, 0].bar(range(len(success_df)), success_df['avg_degree'], color='green', alpha=0.7)
    axes[1, 0].axhline(success_df['avg_degree'].mean(), color='red', linestyle='--',
                       label=f"Mean: {success_df['avg_degree'].mean():.2f}")
    axes[1, 0].set_xlabel('Case Index')
    axes[1, 0].set_ylabel('Average Degree')
    axes[1, 0].set_title('Average Degree per Case')
    axes[1, 0].legend()
    
    # Plot 4: Processing time
    axes[1, 1].bar(range(len(success_df)), success_df['processing_time'], color='purple', alpha=0.7)
    axes[1, 1].axhline(success_df['processing_time'].mean(), color='red', linestyle='--',
                       label=f"Mean: {success_df['processing_time'].mean():.2f}s")
    axes[1, 1].set_xlabel('Case Index')
    axes[1, 1].set_ylabel('Processing Time (s)')
    axes[1, 1].set_title('Processing Time per Case')
    axes[1, 1].legend()
    
    plt.tight_layout()
    
    # Save figure
    fig_path = os.path.join(OUTPUT_DIR, 'processing_summary.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved visualization to: {fig_path}")
    
    plt.show()
else:
    print("No successful cases to visualize!")

---
## 15. List Output Files

In [ ]:
# List all generated files
output_files = sorted(os.listdir(OUTPUT_DIR))

networkx_files = [f for f in output_files if f.endswith('_networkx.pkl')]
pyg_files = [f for f in output_files if f.endswith('_pyg.pkl')]
other_files = [f for f in output_files if not f.endswith('.pkl') or 'feature_names' in f]

print(f"Output directory: {OUTPUT_DIR}")
print("=" * 60)
print(f"\nNetworkX graphs ({len(networkx_files)} files):")
for f in networkx_files[:5]:
    print(f"  • {f}")
if len(networkx_files) > 5:
    print(f"  ... and {len(networkx_files) - 5} more")

print(f"\nPyG data ({len(pyg_files)} files):")
for f in pyg_files[:5]:
    print(f"  • {f}")
if len(pyg_files) > 5:
    print(f"  ... and {len(pyg_files) - 5} more")

print(f"\nOther files:")
for f in other_files:
    print(f"  • {f}")

---
## 16. How to Load Saved Graphs

In [1]:
def load_graph(filename, graph_dir, graph_type='pyg'):
    """
    Load a saved graph by original filename.
    
    Args:
        filename: Original CSV filename (with or without .csv extension)
                  Example: 'Co_GNN_Inps_0010007803' or 'Co_GNN_Inps_0010007803.csv'
        graph_dir: Directory containing saved graphs
        graph_type: 'pyg' or 'networkx'
    
    Returns:
        Loaded graph object
    """
    # Remove .csv extension if present
    if filename.endswith('.csv'):
        filename = filename[:-4]
    
    if graph_type == 'pyg':
        path = os.path.join(graph_dir, f"{filename}_pyg.pkl")
    else:
        path = os.path.join(graph_dir, f"{filename}_networkx.pkl")
    
    with open(path, 'rb') as f:
        graph = pickle.load(f)
    
    return graph


def load_all_graphs(graph_dir, graph_type='pyg'):
    """
    Load all saved graphs.
    
    Returns:
        dict: {filename: graph}
    """
    if graph_type == 'pyg':
        pattern = "*_pyg.pkl"
    else:
        pattern = "*_networkx.pkl"
    
    files = glob.glob(os.path.join(graph_dir, pattern))
    
    graphs = {}
    for f in tqdm(files, desc=f"Loading {graph_type} graphs"):
        basename = os.path.basename(f)
        # Remove suffix to get original name
        if graph_type == 'pyg':
            name = basename.replace('_pyg.pkl', '')
        else:
            name = basename.replace('_networkx.pkl', '')
        
        with open(f, 'rb') as file:
            graphs[name] = pickle.load(file)
    
    return graphs


print("✓ Loading functions defined")
print("\nExample usage:")
print("-" * 50)
print("# Load single graph by original filename")
print("graph = load_graph('Co_GNN_Inps_0010007803', './graphs', 'pyg')")
print("")
print("# Load all graphs")
print("all_graphs = load_all_graphs('./graphs', 'pyg')")

✓ Loading functions defined

Example usage:
--------------------------------------------------
# Load single graph by original filename
graph = load_graph('Co_GNN_Inps_0010007803', './graphs', 'pyg')

# Load all graphs
all_graphs = load_all_graphs('./graphs', 'pyg')


In [ ]:
# Test loading a graph
if len(results) > 0 and results[0]['status'] == 'success':
    test_filename = results[0]['base_name']
    
    print(f"Testing graph loading for: {test_filename}")
    print("=" * 50)
    
    # Load PyG graph
    if PYTORCH_AVAILABLE:
        pyg_graph = load_graph(test_filename, OUTPUT_DIR, 'pyg')
        print(f"\nPyG graph:")
        print(f"  Nodes: {pyg_graph.num_nodes:,}")
        print(f"  Edges: {pyg_graph.num_edges:,}")
        print(f"  Features: {pyg_graph.x.shape}")
    
    # Load NetworkX graph
    nx_graph = load_graph(test_filename, OUTPUT_DIR, 'networkx')
    print(f"\nNetworkX graph:")
    print(f"  Nodes: {nx_graph.number_of_nodes():,}")
    print(f"  Edges: {nx_graph.number_of_edges():,}")
    
    print("\n✓ Loading test successful!")

---
## 17. Verify Node Features for Sample Graph

In [ ]:
# ============================================================
# VERIFY NODE FEATURES FOR ONE SAMPLE GRAPH
# ============================================================

if len(results) > 0 and results[0]['status'] == 'success':
    sample_filename = results[0]['base_name']
    
    print(f"Verifying features for: {sample_filename}")
    print("=" * 60)
    
    # Load PyG graph
    pyg_path = os.path.join(OUTPUT_DIR, f"{sample_filename}_pyg.pkl")
    with open(pyg_path, 'rb') as f:
        data = pickle.load(f)
    
    # Load feature names
    with open(os.path.join(OUTPUT_DIR, 'feature_names.pkl'), 'rb') as f:
        feature_names = pickle.load(f)
    
    print(f"\n1. BASIC INFO")
    print(f"   Nodes: {data.num_nodes:,}")
    print(f"   Edges: {data.num_edges:,}")
    print(f"   Feature tensor shape: {data.x.shape}")
    
    print(f"\n2. FEATURE COLUMNS ({len(feature_names)} total)")
    for i, name in enumerate(feature_names):
        print(f"   [{i:2d}] {name}")
    
    # Convert to numpy
    features_tensor = data.x.numpy() if hasattr(data.x, 'numpy') else data.x
    
    print(f"\n3. VERIFY NORMALIZATION (Numerical Features)")
    print(f"   {'Feature':<20} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
    print("   " + "-" * 60)
    
    for i in range(min(7, len(feature_names))):  # First 7 are numerical
        col_data = features_tensor[:, i]
        print(f"   {feature_names[i]:<20} {np.mean(col_data):>10.4f} {np.std(col_data):>10.4f} {np.min(col_data):>10.4f} {np.max(col_data):>10.4f}")
    
    print("\n   ✓ If Mean ≈ 0 and Std ≈ 1, normalization is correct!")
    
    print(f"\n4. VERIFY ONE-HOT ENCODING (Categorical Features)")
    for i in range(7, len(feature_names)):
        col_data = features_tensor[:, i]
        count_ones = np.sum(col_data == 1)
        count_zeros = np.sum(col_data == 0)
        print(f"   {feature_names[i]}: {count_ones:,} ones, {count_zeros:,} zeros")

---
## 18. Final Summary

In [ ]:
print("\n" + "=" * 70)
print("  BATCH PROCESSING COMPLETE!")
print("=" * 70)

success_count = len(summary_df[summary_df['status'] == 'success'])
failed_count = len(summary_df[summary_df['status'] == 'failed'])

print(f"""
SUMMARY
{'─' * 60}

Cases Processed:
  • Total:       {len(file_pairs)}
  • Successful:  {success_count}
  • Failed:      {failed_count}

Graph Statistics:
  • Nodes:       {success_df['num_nodes'].mean():,.0f} (avg per case)
  • Edges:       {success_df['num_edges'].mean():,.0f} (avg per case)
  • Features:    {success_df['num_features'].iloc[0] if len(success_df) > 0 else 'N/A'}
  • Avg degree:  {success_df['avg_degree'].mean():.2f}

Output Files ({OUTPUT_DIR}/):
  • <original_name>_networkx.pkl  ({success_count} files)
  • <original_name>_pyg.pkl       ({success_count} files)
  • all_graphs_summary.csv
  • feature_names.pkl
  • processing_summary.png

Next Steps:
  1. Load graphs: all_graphs = load_all_graphs('{OUTPUT_DIR}', 'pyg')
  2. Train GNN model across all cases
  3. Evaluate and compare results
""")